In [1]:
import sys
import os

# Add the DPVO project root to the Python path (parent of andy/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added {project_root} to sys.path")


Added /home/campus.ncl.ac.uk/c4071391/Projects/DPVO to sys.path


In [2]:
import collections

import torch
import torch.nn as nn
import onnx

from dpvo.net import VONet

# --- 1. Load the DPVO PyTorch Model ---
# Weights path: project root dpvo.pth or andy/dpvo.pth
pth_model_path = os.path.join(project_root, "dpvo.pth")
if not os.path.isfile(pth_model_path):
    pth_model_path = os.path.join(project_root, "andy", "dpvo.pth")
assert os.path.isfile(pth_model_path), f"Checkpoint not found: {pth_model_path}"

export_device = torch.device("cpu")

model = VONet()

ckpt = torch.load(pth_model_path, map_location="cpu", weights_only=True)
state_dict = ckpt if isinstance(ckpt, dict) and "state_dict" not in ckpt else ckpt.get("state_dict", ckpt)

# Strip DataParallel prefix; drop update.lmbda if present (DPVO load_weights does this)
new_state_dict = collections.OrderedDict()
for k, v in state_dict.items():
    if "update.lmbda" in k:
        continue
    new_state_dict[k.replace("module.", "")] = v

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval().to(export_device)



ModuleNotFoundError: No module named 'onnx_mods'

In [ ]:

# --- 3. Export DPVO encoder sub-models (fnet, inet) to ONNX ---
# DPVO Update uses custom ops (fastba.neighbors, torch_scatter) so it stays PyTorch-only.
# Only the feature encoders (fnet, inet) from Patchifier are exported.

# from dpvo.utils import patchify # think this was here by mistake


onnx_dir = os.path.join(project_root, "andy", "onnx")
os.makedirs(onnx_dir, exist_ok=True)
fnet_onnx_path = os.path.join(onnx_dir, "fnet.onnx")
inet_onnx_path = os.path.join(onnx_dir, "inet.onnx")
patchify_onnx_path = os.path.join(onnx_dir, "patchify.onnx")
update_onnx_path = os.path.join(onnx_dir, "update.onnx")

OPSET = 17


device = next(model.parameters()).device
# device='cuda'
print(f'Device: {device}')

# Input: images [B, N, 3, H, W] already normalized as in DPVO: 2*(x/255)-0.5
# Output: fmap [B, N, 128, H/4, W/4], imap [B, N, 384, H/4, W/4] (Patchifier then divides by 4.0 in code)
fnet_export = model.patchify.fnet.to(device).eval()
inet_export = model.patchify.inet.to(device).eval()
patchify_export = model.patchify.to(device).eval()
update_export = model.update.to(device).eval()



Device: cuda:0


In [ ]:


# --- 2. Create dummy inputs for ONNX tracing ---
# DPVO Patchifier uses fnet and inet with images [B, N, 3, H, W] normalized as 2*(x/255)-0.5.
# BasicEncoder4 has total stride 4 -> feature map at H/4, W/4.

BATCH = 1
NUM_FRAMES = 1
C = 3
H = 480
W = 640


# Raw image range [0, 255] (same as DPVO before normalization)
images_raw = (torch.rand(BATCH, NUM_FRAMES, C, H, W, device=device) * 255.0).to(torch.float32)
# Normalized as in DPVO: 2 * (image / 255.0) - 0.5
images = (2.0 * (images_raw / 255.0) - 0.5).to(torch.float32)

print('Exporting fnet (feature encoder)...')
torch.onnx.export(
    fnet_export,
    (images,),
    fnet_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['images'],
    output_names=['fmap'],
    dynamic_axes={
        'images': {0: 'batch', 1: 'frames', 3: 'height', 4: 'width'},
        'fmap': {0: 'batch', 1: 'frames', 3: 'h4', 4: 'w4'},
    },
)
print(f'Saved: {fnet_onnx_path}')

print('Exporting inet (context encoder)...')
torch.onnx.export(
    inet_export,
    (images,),
    inet_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['images'],
    output_names=['imap'],
    dynamic_axes={
        'images': {0: 'batch', 1: 'frames', 3: 'height', 4: 'width'},
        'imap': {0: 'batch', 1: 'frames', 3: 'h4', 4: 'w4'},
    },
)


Exporting fnet (feature encoder)...
Saved: /home/campus.ncl.ac.uk/c4071391/Projects/DPVO/andy/onnx/fnet.onnx
Exporting inet (context encoder)...


/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/dpvo/lib/python3.9/site-packages/torch/onnx/symbolic_helper.py:1454: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  warnings.warn(


In [3]:
# dummy inputs for Patchifier
patch_size = 3
patches_per_image = 96

print('Exporting patchify...')
torch.onnx.export(
    patchify_export,
    (images, patches_per_image,),
    patchify_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['images', 'patches_per_image'],
    output_names=['fmap', 'gmap', 'imap', 'patches', 'index', 'clr'],
    dynamic_axes={
        'images': {0: 'batch', 1: 'frames', 3: 'height', 4: 'width'},
        'fmap': {0: 'batch', 1: 'frames', 3: 'h4', 4: 'w4'},
        'imap': {0: 'batch', 1: 'frames', 3: 'h4', 4: 'w4'},
        'gmap': {0: 'batch', 1: 'frames', 3: 'h4', 4: 'w4'},
        'patches': {0: 'batch', 1: 'patches_per_image', 2: 'Channels', 3: 'P', 4: 'P'},
        'index': {0: 'index'},
        'clr': {0: 'batch', 1: 'patch_index', 2: 'channel'}
    },
)


Exporting patchify...


NameError: name 'patchify_export' is not defined

In [ ]:
# dummy inputs for update
# self, net, inp, corr, flow, ii, jj, kk
# self.pg.net, (delta, weight, _) = \
# self.network.update(self.pg.net, ctx, corr, None, self.pg.ii, self.pg.jj, self.pg.kk)
# setups
pmem = mem = 36
DIM = 384
RES = 4
P = 3
M = patches_per_image
IN_FEATS = 2 * 49 * P * P # 882
E = number_of_active_edges = 512
# actual inputs
device = 'cuda'
net_dummy  = torch.zeros(1, E, DIM, device=device, dtype=torch.float32)
inp_dummy  = torch.zeros(1, E, DIM, device=device, dtype=torch.float32)
corr_dummy = torch.zeros(1, E, IN_FEATS, device=device, dtype=torch.float32)
flow_dummy = torch.zeros(1, E, 2, device=device, dtype=torch.float32)

ii_dummy = torch.zeros(E, device=device, dtype=torch.long)
jj_dummy = torch.zeros(E, device=device, dtype=torch.long)
kk_dummy = torch.arange(E, device=device, dtype=torch.long) % (M * pmem)    

print('Exporting update...')
torch.onnx.export(
    update_export,
    (net_dummy, inp_dummy, corr_dummy, flow_dummy, ii_dummy, jj_dummy, kk_dummy),
    update_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['net', 'inp', 'corr', 'flow', 'ii', 'jj', 'kk'],
    output_names=['net', 'net_out'],
    dynamic_axes={
        'net': {1: 'num_edges',},
        'inp': {1: 'num_edges',},
        'corr': {1: 'num_edges',},
        'flow': {1: 'num_edges',},
        'ii': {0: 'num_edges',},
        'jj': {0: 'num_edges',},
        'kk': {0: 'num_edges',},

    },
)
print(f'Saved: {fnet_onnx_path}')
print(f'Saved: {inet_onnx_path}')
print(f'Saved: {patchify_onnx_path}')
print(f'Saved: {update_onnx_path}')

# --- 4. Verify ONNX files ---
for name, path in [('fnet', fnet_onnx_path), ('inet', inet_onnx_path), ('patchifier', patchify_onnx_path), ('update', update_onnx_path)]:
    m = onnx.load(path)
    onnx.checker.check_model(m)
    print(f'ONNX model check passed: {name}')

print('Done. Use --backend onnx with onnx_dir=andy/onnx to run with PyTorch+ONNX.')



Exporting update...


UnsupportedOperatorError: ONNX export failed on an operator with unrecognized namespace torch_scatter::scatter_max. If you are trying to export a custom operator, make sure you registered it with the right domain and version.

In [ ]:
import onnx
from onnx import numpy_helper

model = onnx.load("onnx/fnet.onnx")

dtypes = set()

for tensor in model.graph.initializer:
    dtypes.add(tensor.data_type)

print("Tensor data types:", dtypes)

Tensor data types: {1}


| ID | Meaning           |
| -- | ----------------- |
| 1  | float32 (normal)  |
| 10 | float16           |
| 3  | int8  ← quantised |
| 2  | uint8 ← quantised |
